# Q1 Source Probe

Audits the official SEC evidence layer, accession-level facts, concept mapping, latest-restated selection, and reconciliation to the frozen analytical release.

Data as of: 2024-01-28.

In [1]:
from pathlib import Path
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
if ROOT.name == 'notebooks':
    ROOT = ROOT.parent
pd.set_option('display.max_columns', 100)

In [2]:
universe = pd.read_csv(ROOT / 'data/reference/company_universe.csv')
events = pd.read_csv(ROOT / 'data/reference/events.csv')
concept_map = pd.read_csv(ROOT / 'data/reference/concept_map.csv')
facts = pd.read_csv(ROOT / 'data/normalized/financial_facts.csv')
latest = pd.read_csv(ROOT / 'data/processed/sec_latest_restated_long.csv')
reconciliation = pd.read_csv(ROOT / 'data/processed/sec_manual_reconciliation.csv')
display(universe[['ticker', 'status_group', 'analysis_scope_group', 'q1_release_included', 'q2_event_candidate']])
display(events[['event_id', 'company_id', 'event_type', 'event_date', 'qualifies_for_q2']])
display(concept_map[['canonical_field', 'source_tag', 'sign_multiplier', 'required_for_q1']])

,ticker,status_group,analysis_scope_group,q1_release_included,q2_event_candidate
0,FTCH,acquired,historical_reference,0,1
1,POSH,acquired,historical_reference,0,1
2,ABNB,active,q1_candidate,0,0
3,BYON,active,q1_candidate,0,0
4,EXPE,active,q1_candidate,0,0
5,REAL,active,q1_candidate,0,1
6,RVLV,active,q1_candidate,0,0
7,SFIX,active,q1_candidate,0,1
8,W,active,q1_candidate,0,0
9,AMZN,active,q1_release,1,0


,event_id,company_id,event_type,event_date,qualifies_for_q2
0,EVT-FTCH-2023-01,ftch,emergency_financing_asset_exit,2023-12-18,0
1,EVT-GRPN-2023-01,grpn,going_concern,2023-03-16,0
2,EVT-REAL-2024-01,real,debt_exchange,2024-02-29,0
3,EVT-SFIX-2023-01,sfix,restructuring_candidate,2023-01-05,0
4,EVT-POSH-2023-01,posh,acquisition_exit_candidate,2023-01-05,0


,canonical_field,source_tag,sign_multiplier,required_for_q1
0,revenue,RevenueFromContractWithCustomerExcludingAssess...,1,1
1,gross_profit,GrossProfit,1,0
2,operating_income,OperatingIncomeLoss,1,1
3,net_income,NetIncomeLoss|ProfitLoss|NetIncomeLossAvailabl...,1,1
4,total_assets,Assets,1,1
5,total_liabilities,Liabilities,1,1
6,total_equity,StockholdersEquity|StockholdersEquityIncluding...,1,1
7,current_assets,AssetsCurrent,1,0
8,current_liabilities,LiabilitiesCurrent,1,0
9,cash_and_equivalents,CashAndCashEquivalentsAtCarryingValue,1,0


In [3]:
probe = pd.DataFrame({
    'normalized_facts': facts.groupby('ticker').size(),
    'latest_canonical_facts': latest.groupby('ticker').size(),
    'manual_matches': reconciliation.query("reconciliation_status == 'match'").groupby('ticker').size(),
    'mapping_reviews': reconciliation.query("reconciliation_status == 'review_company_mapping'").groupby('ticker').size(),
}).fillna(0).astype(int)
display(probe)

,normalized_facts,latest_canonical_facts,manual_matches,mapping_reviews
ticker,,,,
AMZN,104,39,39,0
BKNG,92,36,36,0
CHWY,98,39,38,1
DASH,107,33,30,3
EBAY,101,39,39,0
ETSY,97,36,33,3


Official SEC companyfacts and submissions JSON are cached for all six release companies. Accession and filing-date history is retained in the normalized layer. Mapping differences remain explicit review items and do not silently overwrite the manually reconciled Q1 analytical mart.